# 05b - Game Predictions (Binary Ensemble)

Predict full game statlines for starting pitchers using the binary model ensemble.

**Key differences from 05:**
- Uses 7 separate binary classifiers instead of one 7-class model
- Each outcome model has its own optimized features
- Should have better calibration for elite pitchers

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from datetime import date, timedelta
from pathlib import Path

from src.game_predictor_binary import GamePredictorBinary, format_prediction_summary
from src.data.mlb_api import get_games_with_lineups, check_lineup_status

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

/Applications/miniconda3/envs/mlbenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Memory management utilities
import gc
import psutil

def clear_mem():
    """Run garbage collection and print memory usage."""
    gc.collect()
    mem_gb = psutil.Process().memory_info().rss / 1024**3
    print(f"Memory: {mem_gb:.2f} GB")
    return mem_gb

clear_mem()

Memory: 0.27 GB


0.273193359375

## 1. Initialize Predictor

In [3]:
# Initialize predictor with binary ensemble
predictor = GamePredictorBinary(
    ensemble_dir='../models/binary_ensemble',
    preprocessor_path='../models/matchup_preprocessor.pkl',
    pitcher_profiles_path='../data/profiles/pitcher_arsenal.csv',
    batter_profiles_path='../data/profiles/batter_profiles.csv',
    pitcher_rolling_path='../data/profiles/pitcher_rolling.csv',
    batter_rolling_path='../data/profiles/batter_rolling.csv',
    park_factors_path= "../data/profiles/park_factors.csv"
)

print(f"Feature names: {len(predictor.feature_names)}")
print(f"Outcome classes: {predictor.outcome_classes}")

Feature names: 207
Outcome classes: ['1B', '2B', '3B', 'BB', 'HR', 'K', 'OUT']


## 2. Check Today's Games

In [4]:
# Check lineup availability
today = date.today().strftime("%Y-%m-%d")
print(f"Checking games for {today}...")

status = check_lineup_status(today)
print(f"\nTotal games: {status['total_games']}")
print(f"With lineups: {status['games_with_lineups']}")
print(f"With probable pitchers: {status['games_with_probable_pitchers']}")

print("\nGames:")
for game in status["games"]:
    away = game["away_team"]["abbreviation"]
    home = game["home_team"]["abbreviation"]
    away_p = game["away_pitcher"]["pitcher_name"] if game["away_pitcher"] else "TBD"
    home_p = game["home_pitcher"]["pitcher_name"] if game["home_pitcher"] else "TBD"
    lineups = "Lineups" if game["lineups_available"] else "No lineups"
    print(f"  {away} @ {home}: {away_p} vs {home_p} [{lineups}]")

Checking games for 2026-04-11...

Total games: 15
With lineups: 8
With probable pitchers: 15

Games:
  AZ @ PHI: Brandon Pfaadt vs Taijuan Walker [Lineups]
  MIA @ DET: Janson Junk vs Casey Mize [Lineups]
  PIT @ CHC: Braxton Ashcraft vs Edward Cabrera [Lineups]
  MIN @ TOR: Joe Ryan vs Eric Lauer [Lineups]
  LAA @ CIN: George Klassen vs Brandon Williamson [Lineups]
  ATH @ NYM: Jacob Lopez vs Kodai Senga [Lineups]
  CWS @ KC: Erick Fedde vs Michael Wacha [Lineups]
  NYY @ TB: Max Fried vs Nick Martinez [Lineups]
  WSH @ MIL: Foster Griffin vs Kyle Harrison [No lineups]
  SF @ BAL: Logan Webb vs Chris Bassitt [No lineups]
  BOS @ STL: Ranger Suarez vs Kyle Leahy [No lineups]
  CLE @ ATL: Parker Messick vs Martín Pérez [No lineups]
  COL @ SD: Ryan Feltner vs Germán Márquez [No lineups]
  TEX @ LAD: Jack Leiter vs Emmet Sheehan [No lineups]
  HOU @ SEA: Lance McCullers Jr. vs Luis Castillo [No lineups]


## 3. Run Predictions

In [5]:
# Predict all games for today
predictions = predictor.predict_day(today)

print(f"\n{'='*60}")
print(f"PREDICTIONS FOR {today}")
print(f"{'='*60}")

for game in predictions:
    print(f"\n{game['away_team']} @ {game['home_team']} ({game['status']})")
    print("-" * 40)
    
    if game["away_prediction"]:
        print(format_prediction_summary(game["away_prediction"]))
    
    if game["home_prediction"]:
        print(format_prediction_summary(game["home_prediction"]))
    
    if game["errors"]:
        print(f"  Errors: {game['errors']}")


PREDICTIONS FOR 2026-04-11

AZ @ PHI (complete)
----------------------------------------
Brandon Pfaadt
  Target IP: 5.2, Sim IP: 5.3
  K: 5.5, BB: 2.0, H: 5.3, HR: 0.6
  Expected Runs: 2.32
  ERA (this start): 3.92
Taijuan Walker
  Target IP: 5.0, Sim IP: 5.0
  K: 4.9, BB: 1.9, H: 4.8, HR: 0.8
  Expected Runs: 2.36
  ERA (this start): 4.25

MIA @ DET (complete)
----------------------------------------
Janson Junk
  Target IP: 5.0, Sim IP: 5.0
  K: 5.0, BB: 1.7, H: 4.6, HR: 0.7
  Expected Runs: 1.89
  ERA (this start): 3.40
Casey Mize
  Target IP: 5.3, Sim IP: 5.3
  K: 5.5, BB: 1.7, H: 5.1, HR: 0.6
  Expected Runs: 2.01
  ERA (this start): 3.40

PIT @ CHC (complete)
----------------------------------------
Braxton Ashcraft
  Target IP: 4.9, Sim IP: 5.0
  K: 5.1, BB: 1.6, H: 4.9, HR: 0.6
  Expected Runs: 2.00
  ERA (this start): 3.60
Edward Cabrera
  Target IP: 5.1, Sim IP: 5.0
  K: 5.1, BB: 2.1, H: 4.7, HR: 0.8
  Expected Runs: 2.33
  ERA (this start): 4.20

MIN @ TOR (complete)
-----

## 4. Detailed View - Single Game

In [6]:
# Show detailed predictions for first complete game
complete_games = [g for g in predictions if g['status'] == 'complete']

if complete_games:
    game = complete_games[0]
    print(f"Detailed view: {game['away_team']} @ {game['home_team']}")
    print("=" * 60)
    
    for side, pred in [("AWAY", game["away_prediction"]), ("HOME", game["home_prediction"])]:
        if pred:
            print(f"\n{side}: {pred['pitcher_name']}")
            print(f"Expected BF: {pred['expected_bf']}")
            print(f"\nExpected Stats:")
            for stat, val in pred['expected_stats'].items():
                print(f"  {stat}: {val}")
            
            print(f"\nPer-Batter Predictions (first time through):")
            for bp in pred['batter_predictions'][:9]:
                k_prob = bp['probabilities']['K']
                bb_prob = bp['probabilities']['BB']
                hr_prob = bp['probabilities']['HR']
                print(f"  {bp['batter_name'][:20]:20s}: K={k_prob:.1%}, BB={bb_prob:.1%}, HR={hr_prob:.1%}")
else:
    print("No complete games available")

Detailed view: AZ @ PHI

AWAY: Brandon Pfaadt
Expected BF: 24.0

Expected Stats:
  K: 5.5
  BB: 2.04
  H: 5.34
  1B: 3.63
  2B: 1.09
  3B: 0.06
  HR: 0.56
  ER: 2.32
  IP_approx: 5.3
  ERA_approx: 3.92

Per-Batter Predictions (first time through):
  Trea Turner         : K=23.0%, BB=8.2%, HR=3.2%
  Kyle Schwarber      : K=24.0%, BB=8.6%, HR=3.6%
  Bryce Harper        : K=22.9%, BB=8.3%, HR=3.4%
  Brandon Marsh       : K=23.5%, BB=8.2%, HR=3.2%
  Bryson Stott        : K=22.5%, BB=8.3%, HR=3.2%
  Adolis García       : K=23.0%, BB=8.1%, HR=3.3%
  J.T. Realmuto       : K=23.1%, BB=8.2%, HR=3.2%
  Alec Bohm           : K=22.5%, BB=8.2%, HR=3.2%
  Justin Crawford     : K=22.9%, BB=8.3%, HR=2.9%

HOME: Taijuan Walker
Expected BF: 23.5

Expected Stats:
  K: 4.9
  BB: 1.85
  H: 4.77
  1B: 2.95
  2B: 0.92
  3B: 0.06
  HR: 0.84
  ER: 2.36
  IP_approx: 5.0
  ERA_approx: 4.25

Per-Batter Predictions (first time through):
  Ketel Marte         : K=22.8%, BB=8.6%, HR=3.4%
  Corbin Carroll      : K=22

## 5. Compare with Yesterday (Completed Games)

In [7]:
# Get yesterday's games (should have lineups from boxscore)
yesterday = (date.today() - timedelta(days=1)).strftime("%Y-%m-%d")
print(f"Predictions for yesterday ({yesterday}):")

yesterday_preds = predictor.predict_day(yesterday)

for game in yesterday_preds:
    if game['status'] == 'complete':
        print(f"\n{game['away_team']} @ {game['home_team']}")
        
        if game["away_prediction"]:
            p = game["away_prediction"]
            s = p["expected_stats"]
            print(f"  {p['pitcher_name']}: {s['K']:.1f} K, {s['BB']:.1f} BB, {s['H']:.1f} H, ERA={s['ERA_approx']}")
        
        if game["home_prediction"]:
            p = game["home_prediction"]
            s = p["expected_stats"]
            print(f"  {p['pitcher_name']}: {s['K']:.1f} K, {s['BB']:.1f} BB, {s['H']:.1f} H, ERA={s['ERA_approx']}")

Predictions for yesterday (2026-04-10):

PIT @ CHC
  Carmen Mlodzinski: 4.3 K, 1.7 BB, 4.5 H, ERA=4.76
  Shota Imanaga: 5.5 K, 1.8 BB, 5.3 H, ERA=4.21

AZ @ PHI
  Michael Soroka: 4.3 K, 1.5 BB, 4.3 H, ERA=3.79
  Jesús Luzardo: 6.2 K, 2.0 BB, 5.6 H, ERA=3.72

MIA @ DET
  Chris Paddack: 4.7 K, 2.0 BB, 4.5 H, ERA=4.15
  Keider Montero: 4.7 K, 1.7 BB, 4.9 H, ERA=4.43

LAA @ CIN
  Jack Kochanowicz: 4.1 K, 1.6 BB, 4.5 H, ERA=4.17
  Chase Burns: 5.9 K, 2.1 BB, 5.2 H, ERA=4.19

MIN @ TOR
  Simeon Woods Richardson: 4.8 K, 1.5 BB, 4.3 H, ERA=3.78
  Patrick Corbin: 4.9 K, 1.7 BB, 4.4 H, ERA=3.58

ATH @ NYM
  J.T. Ginn: 4.3 K, 1.6 BB, 4.0 H, ERA=3.16
  Clay Holmes: 5.0 K, 1.8 BB, 4.4 H, ERA=3.37

NYY @ TB
  Luis Gil: 5.2 K, 1.9 BB, 5.5 H, ERA=4.44
  Steven Matz: 5.0 K, 1.7 BB, 4.5 H, ERA=3.35

SF @ BAL
  Landen Roupp: 5.5 K, 2.1 BB, 5.5 H, ERA=4.98
  Shane Baz: 4.8 K, 1.6 BB, 4.8 H, ERA=3.89

CLE @ ATL
  Slade Cecconi: 4.8 K, 1.7 BB, 5.2 H, ERA=4.16
  Bryce Elder: 5.9 K, 2.3 BB, 6.1 H, ERA=4.3

CW

## 6. Elite Pitcher Check

Verify predictions for elite K pitchers are improved.

In [8]:
# Find elite pitchers in recent predictions
all_preds = yesterday_preds + predictions

elite_k_threshold = 7.0  # Expected Ks
elite_pitchers = []

for game in all_preds:
    for pred_key in ["away_prediction", "home_prediction"]:
        pred = game.get(pred_key)
        if pred and pred["expected_stats"]["K"] >= elite_k_threshold:
            elite_pitchers.append({
                "name": pred["pitcher_name"],
                "K": pred["expected_stats"]["K"],
                "BB": pred["expected_stats"]["BB"],
                "H": pred["expected_stats"]["H"],
                "ERA": pred["expected_stats"]["ERA_approx"],
                "BF": pred["expected_bf"],
            })

if elite_pitchers:
    print(f"Elite K predictions (>= {elite_k_threshold} K):")
    elite_df = pd.DataFrame(elite_pitchers).sort_values("K", ascending=False)
    print(elite_df.to_string(index=False))
else:
    print("No elite K predictions found in recent games")

Elite K predictions (>= 7.0 K):
     name    K   BB    H  ERA   BF
Max Fried 7.12 2.54 6.06 3.49 26.0


## 7. Summary Table

In [9]:
# Create summary DataFrame
rows = []

for game in predictions:
    for side, pred_key in [("away", "away_prediction"), ("home", "home_prediction")]:
        pred = game.get(pred_key)
        if pred:
            s = pred["expected_stats"]
            rows.append({
                "Game": f"{game['away_team']}@{game['home_team']}",
                "Pitcher": pred["pitcher_name"],
                "BF": pred["expected_bf"],
                "K": s["K"],
                "BB": s["BB"],
                "H": s["H"],
                "HR": s["HR"],
                "ER": s["ER"],
                "IP": s["IP_approx"],
                "ERA": s["ERA_approx"],
            })

if rows:
    summary_df = pd.DataFrame(rows)
    print(f"\nPrediction Summary for {today}:")
    print(summary_df.to_string(index=False))
else:
    print("No predictions available")


Prediction Summary for 2026-04-11:
   Game            Pitcher   BF    K   BB    H   HR   ER  IP  ERA
 AZ@PHI     Brandon Pfaadt 24.0 5.50 2.04 5.34 0.56 2.32 5.3 3.92
 AZ@PHI     Taijuan Walker 23.5 4.90 1.85 4.77 0.84 2.36 5.0 4.25
MIA@DET        Janson Junk 22.5 4.96 1.73 4.58 0.74 1.89 5.0 3.40
MIA@DET         Casey Mize 23.0 5.50 1.73 5.11 0.56 2.01 5.3 3.40
PIT@CHC   Braxton Ashcraft 21.0 5.09 1.63 4.89 0.63 2.00 5.0 3.60
PIT@CHC     Edward Cabrera 22.5 5.12 2.12 4.68 0.79 2.33 5.0 4.20
MIN@TOR           Joe Ryan 21.5 5.00 1.80 5.21 0.81 2.33 5.0 4.19
MIN@TOR         Eric Lauer 21.5 4.47 1.56 4.22 0.73 2.01 4.3 4.18
LAA@CIN     George Klassen 16.0 2.66 1.03 2.83 0.40 1.28 2.7 4.33
LAA@CIN Brandon Williamson 23.0 5.53 1.78 5.04 0.65 2.30 5.3 3.88
ATH@NYM        Jacob Lopez 21.0 4.83 1.57 4.45 0.59 1.72 4.7 3.31
ATH@NYM        Kodai Senga 22.0 4.93 1.92 4.93 0.66 2.14 4.7 4.14
 CWS@KC        Erick Fedde 21.5 4.43 1.60 4.47 0.59 2.00 4.3 4.15
 CWS@KC      Michael Wacha 23.0 5.83 2.1